In [0]:
import yaml
import os
from pyspark.sql import functions as F

repo_path = os.getcwd().split("/notebooks/")[0]

config_path = f"{repo_path}/configs/env/dev.yml"

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

CATALOG       = cfg["catalog"]
SILVER_SCHEMA = cfg["silver_schema"]
GOLD_SCHEMA   = cfg["gold_schema"]

SILVER = f"{CATALOG}.{SILVER_SCHEMA}.spotify_tracks"
G      = f"{CATALOG}.{GOLD_SCHEMA}"

print(f"Reading from : {SILVER}")
print(f"Writing to   : {G}.*")

In [0]:
df_silver = spark.table(SILVER)

print(f"Total rows in Silver   : {df_silver.count():,}")
print(f"Distinct track_ids     : {df_silver.select('track_id').distinct().count():,}")
print(f"Distinct genres        : {df_silver.select('track_genre').distinct().count():,}")
print(f"Distinct album names   : {df_silver.select('album_name').distinct().count():,}")

display(
    df_silver
    .select("track_id", "track_name", "artists", "album_name", "track_genre")
    .limit(5)
)

In [0]:
spark.sql(f"""
    MERGE INTO {G}.dim_genre AS target
    USING (
        WITH distinct_genres AS (
            SELECT DISTINCT
                LOWER(TRIM(track_genre)) AS genre_name
            FROM {SILVER}
            WHERE track_genre IS NOT NULL
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY genre_name) AS genre_id,
            genre_name
        FROM distinct_genres
    ) AS source
    ON target.genre_name = source.genre_name
    WHEN NOT MATCHED THEN
        INSERT (genre_id, genre_name)
        VALUES (source.genre_id, source.genre_name)
""")

count = spark.table(f"{G}.dim_genre").count()
print(f"dim_genre rows: {count:,}")
display(spark.table(f"{G}.dim_genre").orderBy("genre_id").limit(10))

In [0]:
spark.sql(f"""
    MERGE INTO {G}.dim_artist AS target
    USING (
        WITH exploded AS (
            SELECT DISTINCT
                TRIM(artist_col.col) AS artist_name
            FROM {SILVER}
            LATERAL VIEW EXPLODE(artists) artist_col -- array to rows
            WHERE artists IS NOT NULL
        ),
        ranked AS (
            SELECT
                ROW_NUMBER() OVER (ORDER BY artist_name) AS artist_id,
                artist_name
            FROM exploded
            WHERE artist_name IS NOT NULL
              AND artist_name != ''
        )
        SELECT artist_id, artist_name
        FROM ranked
    ) AS source
    ON target.artist_name = source.artist_name
    WHEN NOT MATCHED THEN
        INSERT (artist_id, artist_name)
        VALUES (source.artist_id, source.artist_name)
""")

count = spark.table(f"{G}.dim_artist").count()
print(f"dim_artist rows: {count:,}")
display(spark.table(f"{G}.dim_artist").orderBy("artist_id").limit(10))

In [0]:
# First truncate to remove old duplicate rows
spark.sql(f"TRUNCATE TABLE {G}.dim_album")

spark.sql(f"""
    MERGE INTO {G}.dim_album AS target
    USING (
        -- CTE 1: one row per album, pick main artist consistently
        WITH album_artist AS (
            SELECT
                TRIM(album_name)      AS album_name,
                MIN(TRIM(artists[0])) AS main_artist
            FROM {SILVER}
            WHERE album_name IS NOT NULL
            GROUP BY TRIM(album_name)
        ),
        -- CTE 2: assign surrogate key — window function only, no subquery
        ranked AS (
            SELECT
                ROW_NUMBER() OVER (ORDER BY album_name) AS album_id,
                album_name,
                main_artist
            FROM album_artist
        ),
        -- CTE 3: resolve artist_id with a JOIN — no correlated subquery needed
        with_artist_id AS (
            SELECT
                r.album_id,
                r.album_name,
                da.artist_id
            FROM ranked r
            LEFT JOIN {G}.dim_artist da ON da.artist_name = r.main_artist
        )
        SELECT album_id, album_name, artist_id
        FROM with_artist_id
    ) AS source
    ON target.album_name = source.album_name
    WHEN NOT MATCHED THEN
        INSERT (album_id, album_name, artist_id)
        VALUES (source.album_id, source.album_name, source.artist_id)
""")

count = spark.table(f"{G}.dim_album").count()
print(f"dim_album rows: {count:,}")
display(spark.table(f"{G}.dim_album").orderBy("album_id").limit(10))

In [0]:
spark.sql(f"""
    MERGE INTO {G}.dim_audio_features AS target
    USING (
        WITH features AS (
            SELECT
                track_id,
                AVG(danceability)       AS danceability,
                AVG(energy)             AS energy,
                MAX(key)                AS key,
                AVG(loudness)           AS loudness,
                MAX(mode)               AS mode,
                AVG(speechiness)        AS speechiness,
                AVG(acousticness)       AS acousticness,
                AVG(instrumentalness)   AS instrumentalness,
                AVG(liveness)           AS liveness,
                AVG(valence)            AS valence,
                AVG(tempo)              AS tempo,
                MAX(time_signature)     AS time_signature
            FROM {SILVER}
            WHERE track_id IS NOT NULL
            GROUP BY track_id
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY track_id) AS audio_id,
            track_id,
            danceability,
            energy,
            key,
            loudness,
            mode,
            speechiness,
            acousticness,
            instrumentalness,
            liveness,
            valence,
            tempo,
            time_signature
        FROM features
    ) AS source
    ON target.track_id = source.track_id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

count = spark.table(f"{G}.dim_audio_features").count()
print(f"dim_audio_features rows: {count:,}")
display(spark.table(f"{G}.dim_audio_features").limit(10))

In [0]:
dims = ["dim_genre", "dim_artist", "dim_album", "dim_audio_features"]

print(f"{'Table':<30} {'Rows':>10}")
print("-" * 42)
for dim in dims:
    count = spark.table(f"{G}.{dim}").count()
    print(f"{dim:<30} {count:>10,}")

In [0]:
spark.sql(f"""
    SELECT
        COUNT(DISTINCT LOWER(TRIM(s.track_genre)))  AS silver_genres,
        COUNT(DISTINCT dg.genre_id)                 AS dim_genres,
        COUNT(DISTINCT TRIM(s.artists[0]))           AS silver_main_artists,
        COUNT(DISTINCT da.artist_id)                AS dim_artists,
        COUNT(DISTINCT TRIM(s.album_name))           AS silver_albums,
        COUNT(DISTINCT dal.album_id)                AS dim_albums,
        COUNT(DISTINCT s.track_id)                  AS silver_tracks,
        COUNT(DISTINCT daf.audio_id)                AS dim_audio_features
    FROM {SILVER} s
    LEFT JOIN {G}.dim_genre          dg  ON LOWER(TRIM(s.track_genre)) = dg.genre_name
    LEFT JOIN {G}.dim_artist         da  ON TRIM(s.artists[0]) = da.artist_name
    LEFT JOIN {G}.dim_album          dal ON TRIM(s.album_name) = dal.album_name
    LEFT JOIN {G}.dim_audio_features daf ON s.track_id = daf.track_id
""").show()